# Avaliação da qualidade dos pedidos

Neste notebook, verificamos a tabela `olist_orders_dataset.csv` com contratos Pandera. As regras foram escolhidas para esta tabela, considerando que cada registro representa um pedido e que algumas datas ficam vazias enquanto o pedido ainda percorre seu fluxo.

As verificações estão organizadas em cinco grupos:

1. **Completude:** campos essenciais estão preenchidos e os nulos em datas opcionais ficam dentro do limite adotado.
2. **Unicidade:** o identificador do pedido não aparece mais de uma vez.
3. **Validade:** identificadores, estados e representações de data seguem os formatos e valores esperados.
4. **Consistência:** as datas respeitam a ordem do processo do pedido.
5. **Faixa temporal:** as datas estão dentro do período coberto por esta versão do conjunto de dados.

Em cada grupo, `lazy=True` permite reunir as violações antes de exibi-las. O relatório indica quais regras falharam; a interpretação depende do contexto dos dados.

As chamadas e a sintaxe de validação seguem a documentação do [Pandera](https://pandera.readthedocs.io/en/stable/). Os limites e regras abaixo são definidos para a tabela Olist analisada.

In [1]:
import numpy as np
import pandas as pd
import pandera.pandas as pa
from pandera.errors import SchemaErrors

pd.set_option("display.max_colwidth", 60)

RAW_PATH = "../data/raw/olist_orders_dataset.csv"

# Mantemos os valores originais como texto para testar nulos, dom?nios e formatos antes da convers?o.
df_raw = pd.read_csv(RAW_PATH, dtype=str)

DATE_COLS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

# Cópia com as datas parseadas, usada nos testes de consistência e atualidade
df_dates = df_raw.copy()
for col in DATE_COLS:
    df_dates[col] = pd.to_datetime(df_dates[col], errors="coerce")

df_raw.shape

(99441, 8)

In [2]:
results = []


def run_check(dimension: str, schema: pa.DataFrameSchema, data: pd.DataFrame) -> None:
    """Executa as regras da dimens?o e guarda um resumo junto dos casos encontrados."""
    try:
        schema.validate(data, lazy=True)
    except SchemaErrors as exc:
        failures = exc.failure_cases
        n_rows = failures["index"].nunique()
        results.append({"dimensao": dimension, "status": "FALHOU", "n_falhas": n_rows})
        print(f"[{dimension}] FALHOU — {n_rows} linha(s) com violação:")
        display(failures.head(10))
    else:
        results.append({"dimensao": dimension, "status": "OK", "n_falhas": 0})
        print(f"[{dimension}] OK — nenhuma violação encontrada.")

## 1. Completude

Os identificadores, o estado do pedido e as datas necessárias para a análise precisam estar preenchidos. Já as datas de aprovação e entrega podem estar ausentes quando o pedido não chegou a essas etapas.

Para esses campos opcionais, adotamos um limite de ausência inferior a 5%. O teste verifica a taxa na coluna inteira; se o valor atingir 5% ou ultrapassá-lo, a regra falha. Esse percentual é uma escolha para esta análise e pode ser revisto com base no processo e nos resultados observados.

In [3]:
def taxa_de_nulos_abaixo_de(limite: float) -> pa.Check:
    return pa.Check(
        lambda s: s.isna().mean() < limite,
        ignore_na=False,
        name=f"taxa_de_nulos_abaixo_de_{int(limite * 100)}pct",
    )


completeness_schema = pa.DataFrameSchema(
    {
        "order_id": pa.Column(str, nullable=False),
        "customer_id": pa.Column(str, nullable=False),
        "order_status": pa.Column(str, nullable=False),
        "order_purchase_timestamp": pa.Column(str, nullable=False),
        "order_estimated_delivery_date": pa.Column(str, nullable=False),
        "order_approved_at": pa.Column(str, nullable=True, checks=taxa_de_nulos_abaixo_de(0.05)),
        "order_delivered_carrier_date": pa.Column(
            str, nullable=True, checks=taxa_de_nulos_abaixo_de(0.05)
        ),
        "order_delivered_customer_date": pa.Column(
            str, nullable=True, checks=taxa_de_nulos_abaixo_de(0.05)
        ),
    },
    strict=False,
)

run_check("Completude", completeness_schema, df_raw)

[Completude] OK — nenhuma violação encontrada.


## 2. Unicidade

A tabela tem granularidade de um registro por pedido. Por isso, `order_id` deve funcionar como identificador exclusivo: se houver repetições, vale investigar se são duplicatas ou se a granularidade esperada da fonte mudou.

In [4]:
uniqueness_schema = pa.DataFrameSchema(
    {
        "order_id": pa.Column(str, unique=True),
    },
    strict=False,
)

run_check("Unicidade", uniqueness_schema, df_raw)

[Unicidade] OK — nenhuma violação encontrada.


## 3. Validade

Aqui conferimos se os valores têm a forma esperada para esta fonte:

- `order_id` e `customer_id` são representados por 32 caracteres hexadecimais em minúsculas;
- `order_status` pertence ao conjunto de estados aceitos neste contrato;
- as datas preenchidas seguem o padrão textual `AAAA-MM-DD HH:MM:SS`.

Esses testes verificam o formato e o domínio definidos acima. Uma data que combina com o padrão ainda pode precisar de uma verificação de calendário ou de regra de negócio.

In [5]:
ID_PATTERN = r"^[0-9a-f]{32}$"
TIMESTAMP_PATTERN = r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"
VALID_STATUSES = [
    "delivered",
    "shipped",
    "canceled",
    "unavailable",
    "invoiced",
    "processing",
    "created",
    "approved",
]

validity_schema = pa.DataFrameSchema(
    {
        "order_id": pa.Column(str, pa.Check.str_matches(ID_PATTERN)),
        "customer_id": pa.Column(str, pa.Check.str_matches(ID_PATTERN)),
        "order_status": pa.Column(str, pa.Check.isin(VALID_STATUSES)),
        "order_purchase_timestamp": pa.Column(str, pa.Check.str_matches(TIMESTAMP_PATTERN)),
        "order_approved_at": pa.Column(str, pa.Check.str_matches(TIMESTAMP_PATTERN), nullable=True),
        "order_delivered_carrier_date": pa.Column(
            str, pa.Check.str_matches(TIMESTAMP_PATTERN), nullable=True
        ),
        "order_delivered_customer_date": pa.Column(
            str, pa.Check.str_matches(TIMESTAMP_PATTERN), nullable=True
        ),
        "order_estimated_delivery_date": pa.Column(str, pa.Check.str_matches(TIMESTAMP_PATTERN)),
    },
    strict=False,
)

run_check("Validade", validity_schema, df_raw)

[Validade] OK — nenhuma violação encontrada.


## 4. Consistência

Além de validar cada coluna isoladamente, comparamos datas relacionadas ao andamento do pedido. Quando as duas datas envolvidas estão disponíveis, esperamos que:

- a aprovação ocorra na data da compra ou depois dela;
- a postagem à transportadora ocorra na mesma data da aprovação ou depois dela;
- a entrega ao cliente aconteça na mesma data da postagem ou depois dela.

Também aplicamos uma regra condicional: pedidos com estado `delivered` precisam ter a data de entrega preenchida. Nulos fora dessa situação são tratados pelas regras de completude, pois podem corresponder a etapas ainda não concluídas.

In [6]:
def aprovacao_apos_compra(df: pd.DataFrame) -> pd.Series:
    mask = df["order_approved_at"].notna()
    ok = np.where(mask, df["order_approved_at"] >= df["order_purchase_timestamp"], True)
    return pd.Series(ok, index=df.index)


def postagem_apos_aprovacao(df: pd.DataFrame) -> pd.Series:
    mask = df["order_delivered_carrier_date"].notna() & df["order_approved_at"].notna()
    ok = np.where(mask, df["order_delivered_carrier_date"] >= df["order_approved_at"], True)
    return pd.Series(ok, index=df.index)


def entrega_apos_postagem(df: pd.DataFrame) -> pd.Series:
    mask = df["order_delivered_customer_date"].notna() & df["order_delivered_carrier_date"].notna()
    ok = np.where(
        mask, df["order_delivered_customer_date"] >= df["order_delivered_carrier_date"], True
    )
    return pd.Series(ok, index=df.index)


def status_entregue_tem_data_entrega(df: pd.DataFrame) -> pd.Series:
    mask = df["order_status"] == "delivered"
    ok = np.where(mask, df["order_delivered_customer_date"].notna(), True)
    return pd.Series(ok, index=df.index)


consistency_schema = pa.DataFrameSchema(
    columns={
        "order_status": pa.Column(str),
        "order_purchase_timestamp": pa.Column("datetime64[ns]"),
        "order_approved_at": pa.Column("datetime64[ns]", nullable=True),
        "order_delivered_carrier_date": pa.Column("datetime64[ns]", nullable=True),
        "order_delivered_customer_date": pa.Column("datetime64[ns]", nullable=True),
    },
    checks=[
        pa.Check(aprovacao_apos_compra, element_wise=False, name="aprovacao_apos_compra"),
        pa.Check(postagem_apos_aprovacao, element_wise=False, name="postagem_apos_aprovacao"),
        pa.Check(entrega_apos_postagem, element_wise=False, name="entrega_apos_postagem"),
        pa.Check(
            status_entregue_tem_data_entrega,
            element_wise=False,
            name="status_entregue_tem_data_entrega",
        ),
    ],
    strict=False,
)

run_check("Consistência", consistency_schema, df_dates)

[Consistência] FALHOU — 1390 linha(s) com violação:


,schema_context,column,check,check_number,failure_case,index
0,DataFrameSchema,order_id,postagem_apos_aprovacao,1,dcb36b511fcac050b97cd5c05de84dc3,15
7405,DataFrameSchema,order_delivered_carrier_date,postagem_apos_aprovacao,1,2018-07-04 16:17:00,44573
7397,DataFrameSchema,order_delivered_carrier_date,postagem_apos_aprovacao,1,2018-08-23 13:05:00,44167
7398,DataFrameSchema,order_delivered_carrier_date,postagem_apos_aprovacao,1,2018-04-23 22:11:37,44238
7399,DataFrameSchema,order_delivered_carrier_date,postagem_apos_aprovacao,1,2018-07-04 14:31:00,44272
7400,DataFrameSchema,order_delivered_carrier_date,postagem_apos_aprovacao,1,2018-07-05 13:56:00,44295
7401,DataFrameSchema,order_delivered_carrier_date,postagem_apos_aprovacao,1,2018-07-23 10:15:00,44399
7402,DataFrameSchema,order_delivered_carrier_date,postagem_apos_aprovacao,1,2018-04-24 17:04:21,44425
7403,DataFrameSchema,order_delivered_carrier_date,postagem_apos_aprovacao,1,2018-07-04 14:30:00,44452
7404,DataFrameSchema,order_delivered_carrier_date,postagem_apos_aprovacao,1,2017-08-01 15:54:30,44509


## 5. Faixa temporal

Para esta versão do conjunto Olist, consideramos como período coberto o intervalo de 1º de janeiro de 2016 a 31 de dezembro de 2018. O teste verifica se as datas que puderam ser interpretadas estão dentro dessa janela, incluindo seus limites.

Uma data fora do intervalo pode apontar para erro de carga ou valor marcador, mas também pode indicar que a fonte foi atualizada. Antes de alterar a regra, confirme o período efetivamente coberto pelos arquivos utilizados.

In [7]:
WINDOW_START = pd.Timestamp("2016-01-01")
WINDOW_END = pd.Timestamp("2018-12-31")

timeliness_schema = pa.DataFrameSchema(
    {
        col: pa.Column(
            "datetime64[ns]",
            pa.Check.in_range(WINDOW_START, WINDOW_END),
            nullable=True,
        )
        for col in DATE_COLS
    },
    strict=False,
)

run_check("Atualidade", timeliness_schema, df_dates)

[Atualidade] OK — nenhuma violação encontrada.


## Resumo dos testes

A tabela abaixo reúne o resultado de cada dimensão, o estado da validação e a quantidade de linhas sinalizadas. Use os casos exibidos em cada etapa para investigar as violações; o indicador `OK` só significa que as regras configuradas passaram para os dados avaliados.

In [8]:
pd.DataFrame(results)

,dimensao,status,n_falhas
0,Completude,OK,0
1,Unicidade,OK,0
2,Validade,OK,0
3,Consistência,FALHOU,1390
4,Atualidade,OK,0
